In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from transformers import set_seed
from tqdm import tqdm
import pandas as pd
import numpy as np
import torch
import os
os.environ['HF_TOKEN'] = "<hf_token>"
seed = 42
set_seed(seed)

In [ ]:
system_prompt = "You are a patient that has gone to do an interview with a psychologist. The psychologist will ask you a series of questions and you will answer them in a natural way:\n"
user_prompt = "### Input:\n{question}\n\n### Expected Response:\n{answer}"

def apply_prompt(example):
    example["text"] = (
        system_prompt
        + user_prompt.format(question=example["question"], answer=example["answer"])
    )
    return example
#Loading data
dataset_pt = load_dataset('json', data_files='./ordered_PT_test_dataset.json')['train']
dataset_hc = load_dataset('json', data_files='./ordered_healthy_test_dataset.json')['train']
dataset_pt = dataset_pt.map(apply_prompt)
dataset_hc = dataset_hc.map(apply_prompt)
dataset_pt = dataset_pt.to_pandas()
dataset_hc = dataset_hc.to_pandas()
dataset_pt['scz'] = 1
dataset_hc['scz'] = 0
df = pd.concat([dataset_hc, dataset_pt], ignore_index=True)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/778 [00:00<?, ? examples/s]

Map:   0%|          | 0/239 [00:00<?, ? examples/s]

In [ ]:
#Loading models
model_name_pt = "PabloCano1/ordered-PT-gemma3-4b-fine-tuned"
model_name_hc = "PabloCano1/ordered-HC-gemma3-4b-fine-tuned"

print(f"Loading tokenizer")
tokenizer = AutoTokenizer.from_pretrained(model_name_pt)
print(f"Loading models")
model_hc = AutoModelForCausalLM.from_pretrained(
    model_name_hc,
    device_map="cuda",
    dtype="bfloat16",
  attn_implementation="flash_attention_2",
)
model_pt = AutoModelForCausalLM.from_pretrained(
    model_name_pt,
    device_map="cuda",
    dtype="bfloat16",
  attn_implementation="flash_attention_2",
)
model_hc.eval()
model_pt.eval()

def calc_perplexity(df):
    perplexities_hc = []
    perplexities_pt = []
    with torch.no_grad():
        for text in tqdm(df['text'], desc="Calculando perplexities"):
            enc = tokenizer(text, return_tensors="pt").to(model_hc.device)
            outputs_hc = model_hc(**enc, labels=enc["input_ids"])
            loss_hc = outputs_hc.loss.item()  # loss media por token (cross-entropy)
            ppl_hc = float(np.exp(loss_hc))
            perplexities_hc.append(ppl_hc)
            outputs_pt = model_pt(**enc, labels=enc["input_ids"])
            loss_pt = outputs_pt.loss.item()  # loss media por token (cross-entropy)
            ppl_pt = float(np.exp(loss_pt))
            perplexities_pt.append(ppl_pt)
            
    df['per_pt_model'] = perplexities_pt
    df['per_hc_model'] = perplexities_hc
    df['per_pt_model'] = df['per_pt_model'].round(2)
    df['per_hc_model'] = df['per_hc_model'].round(2)

Loading tokenizer
Loading models


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

In [9]:
calc_perplexity(df)

Calculando perplexities: 100%|█████████████████████████████████████████████| 1017/1017 [02:01<00:00,  8.37it/s]


In [ ]:
y_pred = (df['per_pt_model']<df['per_hc_model']).apply(int)

In [13]:
from sklearn.metrics import accuracy_score, classification_report

print(classification_report(df['scz'], y_pred,digits=3))

              precision    recall  f1-score   support

           0      0.560     0.921     0.696       239
           1      0.970     0.778     0.863       778

    accuracy                          0.811      1017
   macro avg      0.765     0.849     0.780      1017
weighted avg      0.873     0.811     0.824      1017

